[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IPNL-POLYU/IPIN-Examples/blob/main/notebooks/indoor_positioning_gallery.ipynb)


# Indoor Positioning Gallery

## Every figure from the SC12 / ION GNSS+ 2026 talk, in one runnable notebook

This notebook strings together the repository-generated figures cited by the SC12 tutorial
deck ("Indoor Navigation as GNSS in Disguise"), **in deck order**, each one reproduced by
importing the repository's own example code — nothing here re-derives an algorithm the
repository already implements. Every section names the deck page it reproduces and the
claim that page makes; the parameters cell above each figure uses the deck's own default
numbers, and you can change them and re-run.

This is not a chapter tutorial — for a guided walk through one chapter at a time, see the
per-chapter notebooks linked from each chapter's README (e.g.
[`ch4_rf_positioning.ipynb`](ch4_rf_positioning.ipynb)). This notebook's job is narrower and
different: **one deck, top to bottom, as code.**

Pinned against `IPIN-Examples` commit `60f57d1` (the deck's own pin); run against `main`, so a
number may have moved since — that is a feature, not a bug, of a notebook over a screenshot.


In [ ]:
# ========================================
# Indoor Positioning Gallery — setup
# ========================================
import os
import sys

GITHUB_REPO = "https://github.com/IPNL-POLYU/IPIN-Examples.git"  # your fork's URL, if you have one

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if os.path.exists("/content/IPIN-Examples/core"):
        os.chdir("/content/IPIN-Examples")
        print("Repository already available.")
    elif GITHUB_REPO:
        print(f"Cloning from {GITHUB_REPO}...")
        get_ipython().system(f"git clone {GITHUB_REPO}")
        os.chdir("/content/IPIN-Examples")
        get_ipython().system("pip install -e . -q")
        print("Setup from GitHub complete!")
    else:
        raise ValueError("GITHUB_REPO not configured.")
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    print(f"Working directory: {os.getcwd()}")

# Every example in this notebook is read-only over data/sim and writes no figures to
# disk on its own (each section calls a `plot_*`/compute function directly and displays
# the returned Figure) -- this redirect is a second, belt-and-suspenders guard in case
# any cell below ever calls a full `main()`-style entry point that saves through
# `core.eval.save_figure`, so a run from inside a real checkout never modifies it.
import tempfile
os.environ.setdefault("IPIN_FIGS_DIR", tempfile.mkdtemp(prefix="ipin_gallery_"))

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams["figure.dpi"] = 100

print("\nSetup complete.")


### A backend note, before the figures start

A handful of the Chapter 4 example modules below call `matplotlib.use("Agg")` at import
time — it is how a command-line script guarantees it never blocks on a GUI window. Imported
into a *notebook*, that call would silently turn inline figures off for every cell after the
first one that imports such a module. This cell imports every module of that kind once, up
front, then re-asserts the inline backend — every subsequent import in this notebook just
looks up an already-loaded module (Python caches imports), so it never re-triggers the
switch, and every figure below renders inline.

In [ ]:
import ch4_rf_point_positioning.example_initial_guess_basin  # noqa: F401
import ch4_rf_point_positioning.example_reflection_ambiguity  # noqa: F401
import ch4_rf_point_positioning.example_closedform_chaining  # noqa: F401

get_ipython().run_line_magic("matplotlib", "inline")
print("Backend:", plt.get_backend())


## p13 — The setup — four anchors, 50 points, one seed

**Deck claim (p13):** one room, one set of test points, one seed — so every later page in
this act compares *physics*, not *tuning*. Every subsequent Chapter 4 figure that runs
"inline" (no `--data`) reuses exactly this scenario.

Source: `generate_scenario(seed=42)` in `ch4_rf_point_positioning/example_comparison.py`.

In [ ]:
from ch4_rf_point_positioning.example_comparison import generate_scenario

SEED = 42  # deck-exact: generate_scenario(seed=42)

anchors, true_positions = generate_scenario(seed=SEED)
print(f"anchors: {len(anchors)}   test points: {len(true_positions)}   area: 10m x 10m")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(anchors[:, 0], anchors[:, 1], marker="^", s=110, c="black", label="anchors", zorder=3)
ax.scatter(true_positions[:, 0], true_positions[:, 1], s=14, c="tab:blue", alpha=0.6, label="test points")
ax.set_aspect("equal")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title(f"Chapter 4 test scenario (seed={SEED})")
ax.legend()
plt.show()


## p14 — Five ways to fix a point indoors — one answers back

**Deck claim (p14):** TOA, RTT, TDOA, AOA and RSS on the same anchors, points and seed,
each with its own physically-meaningful noise ladder. RTT (two-way TOA) and TOA are a
*matched pair* — same anchors, same 50 points, same seed, same noise realisation — so the
gap between their medians is the price of the one thing that differs: TOA carries the
receiver clock as a third unknown `(x, y, c*dt)`, RTT does not `(x, y)`.

Source: `run_inline_comparison()` + `plot_inline_comparison()` in
`ch4_rf_point_positioning/example_comparison.py`.

In [ ]:
from ch4_rf_point_positioning.example_comparison import (
    run_inline_comparison,
    plot_inline_comparison,
)

# run_inline_comparison() takes no parameters -- every one of its defaults (seed=42,
# clock_bias_m=1.5, the five noise ladders) is fixed in the function's own source
# (ch4_rf_point_positioning/example_comparison.py) so this cell's numbers match every
# other inline-comparison figure in this notebook exactly.
level_schedules, results = run_inline_comparison()
fig = plot_inline_comparison(level_schedules, results)
plt.show()


## p18 — Two-way TOA, measured — the offset cancels

**Deck claim (p18):** a round trip is timed on ONE clock, so Eq. (4.7)'s
`d = c(t_rtt - t_proc)/2` carries no unknown receiver-clock offset — convert a distance to
an RTT and back and you recover it exactly.

Source: `range_to_rtt` / `rtt_to_range` (`core/rf/measurement_models.py`), the same pair
`example_toa_positioning.py`'s Example 5 (Eqs. 4.6-4.9) demonstrates.

In [ ]:
from core.rf import range_to_rtt, rtt_to_range, SPEED_OF_LIGHT

DISTANCE_M = 15.0  # deck: "One 15 m link"

print(f"speed of light: {SPEED_OF_LIGHT:.0f} m/s   ->  1 ns of timing = {SPEED_OF_LIGHT * 1e-9 / 2:.3f} m of range")

rtt_ideal = range_to_rtt(DISTANCE_M)
range_back = rtt_to_range(rtt_ideal)
print(f"\ntrue distance:    {DISTANCE_M:.1f} m")
print(f"ideal RTT:        {rtt_ideal * 1e9:.2f} ns")
print(f"range from RTT:   {range_back:.6f} m   (the offset cancels exactly)")


## p19 — But your oscillator did not go away

**Deck claim (p19):** the round trip STILL takes real processing time at the beacon
(Eq. 4.7's `t_proc`). Leave it uncorrected and the whole turnaround reads as extra flight
time — a fixed range error, not noise.

Source: same `range_to_rtt` / `rtt_to_range` pair as p18, with a nonzero
`processing_time`.

In [ ]:
from core.rf import range_to_rtt, rtt_to_range

DISTANCE_M = 15.0        # deck: "The same 15 m link"
PROCESSING_TIME_S = 50e-9  # deck: "+ 50 ns turnaround"

rtt_with_proc = range_to_rtt(DISTANCE_M, processing_time=PROCESSING_TIME_S)
range_uncorrected = rtt_to_range(rtt_with_proc)                                    # ignores t_proc
range_corrected = rtt_to_range(rtt_with_proc, processing_time=PROCESSING_TIME_S)   # accounts for it

print(f"RTT with {PROCESSING_TIME_S * 1e9:.0f} ns turnaround: {rtt_with_proc * 1e9:.2f} ns")
print(f"range, UNCORRECTED:  {range_uncorrected:.2f} m   (error +{range_uncorrected - DISTANCE_M:.2f} m)")
print(f"range, corrected:    {range_corrected:.6f} m")


## p21 — Now put the same four beacons in a corridor

**Deck claim (p21):** the collinear array is not simply "the bad one" — it is bad for
ranges (TOA/TDOA fail every fix from the beacon centroid, which sits on the line of
symmetry) and BETTER than the square for bearings (AOA), because reflecting a position
flips every azimuth. A healthy DOP (1.43 here vs. 1.02 for the square) is necessary, not
sufficient — it does not see the ambiguity that breaks TOA/TDOA.

Source: `compare_geometries()` + `plot_geometry_comparison()` in
`ch4_rf_point_positioning/example_comparison.py` (`--compare-geometry`). This is also the
source of the deck's retypeset two-panel corridor-page figure — no new repository code
was needed to upstream it; the two-panel comparison this function already builds is
exactly what the deck retypesets for slide placement.

In [ ]:
from ch4_rf_point_positioning.example_comparison import (
    compare_geometries,
    plot_geometry_comparison,
)

all_results = compare_geometries(verbose=False)
fig = plot_geometry_comparison(all_results)
plt.show()


## p22 — A local linearization is only as good as its seed

**Deck claim (p22):** sweep the initial guess for AOA over a 41x41 lattice, twice, under
two residual parameterisations. `residual="tan"` fails 785/1681 seeds; the book's default
`residual="angle"` fails 341/1681 — 2.3x fewer, and it removes the QUIET failure class
(stalled-at-seed, converged-wrong-place) completely. The convergence flag is not a check
either way; four failing seeds still walk to nowhere reporting `converged=True`.

Source: `sweep()` + `plot_summary()` in
`ch4_rf_point_positioning/example_initial_guess_basin.py` (the default, no-flags path).

In [ ]:
from ch4_rf_point_positioning.example_initial_guess_basin import sweep, trace_worst, plot_summary

tan_result = sweep("tan")
angle_result = sweep("angle")
worst_seed, worst_history, worst_converged = trace_worst(tan_result)

fig = plot_summary([tan_result, angle_result], (worst_seed, worst_history, worst_converged))
plt.show()


## p23 — …and the basin belongs to the method

**Deck claim (p23) — upstreamed this round.** The sweep above changes AOA's own residual;
this page changes the MEASUREMENT TYPE instead, on the identical 1681-seed lattice and
target. Inside the 10x10 m room every method is essentially fine (3/2/0 failures out of
441 in-room seeds, all exactly ON an anchor). Outside it, AOA has the widest basin **and**
is the only one that never converges to the wrong place — but it also has the most
failures that still report `converged=True` (182 of 341, against 21 each for TOA/TDOA).
The basin belongs to the method, and "converged" is not "correct".

Source: `sweep_arm()` (new) in `ch4_rf_point_positioning/example_initial_guess_basin.py`
— an added `--compare method` option on the SAME example, not a new file. Was
`_build/gen_ch4_basin_methods.py` in the deck workspace; upstreamed with a bit-identical
equivalence test against the residual sweep above
(`tests/ch4_rf_point_positioning/test_initial_guess_basin.py`).

In [ ]:
from core.rf import TDOAPositioner, AOAPositioner, toa_range
from ch4_rf_point_positioning.example_initial_guess_basin import (
    ANCHORS,
    TRUTH,
    METHOD_CLOCK_BIAS_M,
    sweep_arm,
    measurements,
    plot_method_comparison,
    ClockStateSolver,
)

# Deck-exact: same square anchors/target as p23, same 1681-seed lattice, zero noise.
ranges = np.array([toa_range(a, TRUTH) for a in ANCHORS]) + METHOD_CLOCK_BIAS_M  # Eqs. (4.24)-(4.26)
tdoa = ranges[1:] - ranges[:1]

toa_r = sweep_arm(ClockStateSolver(ANCHORS), ranges, "TOA + clock state",
                   title=r"TOA, state $(x,y,c\Delta t)$")
tdoa_r = sweep_arm(TDOAPositioner(ANCHORS, reference_anchor_index=0), tdoa, "TDOA",
                    title="TDOA, ref. anchor 0")
aoa_r = sweep_arm(AOAPositioner(ANCHORS), measurements(), 'AOA, residual="angle"',
                   title=r"AOA, wrap($\psi-\hat{\psi}$)", residual="angle")

for r in (toa_r, tdoa_r, aoa_r):
    n_fail = r["n"] - r["counts"][0]
    print(f"{r['title']:<28} {n_fail}/{r['n']} seeds fail")

fig = plot_method_comparison([toa_r, tdoa_r, aoa_r])
plt.show()


## p24 — Where you start decides which answer you get

**Deck claim (p24) — upstreamed this round.** Same idea, collinear geometry: four
beacons on one line, one target, 1681 seeds. TOA reaches an EXACT solution from 630
seeds — and only 315 of those are the truth; the other 315 are the target's mirror image
about the beacon line, an exact solution of the ranging problem too, and every one of
them reports `converged=True`. TDOA agrees to within one seed in each class. AOA has zero
mirror solves — reflecting a position flips every azimuth — and the widest correct basin
of the three (461), at the price of 1220 loud (never quiet) divergences.

Source: `ch4_rf_point_positioning/example_reflection_ambiguity.py` (new file). Was
`_build/gen_ch4_basin_corridor.py` in the deck workspace; upstreamed as a self-contained
example over the repository's own `data/sim/ch4_rf_2d_linear` dataset.

In [ ]:
from ch4_rf_point_positioning.example_reflection_ambiguity import ARMS, sweep, plot_summary

# ARMS is the same (label, measurement key, solver factory, kwargs) list the example's
# own main() iterates -- TOA+clock state, TDOA, AOA(angle) -- over the corridor's 1681-seed
# lattice, deck-exact.
results = [sweep(*arm, verbose=False) for arm in ARMS]
for r in results:
    c = r["counts"]
    print(f"{r['label']:<24} truth {c[0]:>4}   mirror {c[1]:>4}   fail {c[2]:>4}")

fig = plot_summary(results)
plt.show()


## p25 — No seed at all — closed form first, then refine

**Deck claim (p25) — upstreamed this round.** Fang's closed-form TOA (Eqs. 4.43-4.49) and
Chan's closed-form TDOA (Eqs. 4.50-4.62) take no initial guess at all. On the square room,
closed-form alone already lands within 0.011 m of the fully-refined iterative answer, and
chaining it into the refiner reproduces that answer exactly — the seed problem is not
solved, it is dissolved. On the corridor it does NOT fix the reflection ambiguity: Fang's
own linear system has an exactly singular y-column there (every beacon shares `y=10`), so
every closed-form y-estimate lands at exactly 0.0 — chaining that seed in still converges
on all 100 points, but 50 of them land on the mirror.

Source: `ch4_rf_point_positioning/example_closedform_chaining.py` (new file). Was
`_build/measure_closedform.py` in the deck workspace, itself a relocation of a
session-scratchpad harness; upstreamed self-contained (repository-only imports, no
scratchpad dependency).

In [ ]:
from ch4_rf_point_positioning.example_closedform_chaining import (
    DATASETS,
    load_dataset,
    run_square,
    run_corridor,
    plot_chicken_and_egg,
)

sq_beacons, sq_truth, sq_toa, sq_tdoa, sq_room = load_dataset(DATASETS["square (control)"])
square = run_square(sq_beacons, sq_truth, sq_toa, sq_tdoa, sq_room)

co_beacons, co_truth, co_toa, co_tdoa, co_room = load_dataset(DATASETS["corridor (collinear)"])
corridor = run_corridor(co_beacons, co_truth, co_toa, co_tdoa, co_room)

fig = plot_chicken_and_egg(square, corridor)
plt.show()


## p27 — NLOS, measured — a bias is not noise

**Deck claim (p27):** the controlled experiment — same geometry, noise and seed as the
clean square, except beacons 1 & 2 now carry a fixed bias (+0.8 m range, +18 deg bearing).
AOA fails 10/100. A bias is not noise: no geometry factor (DOP) can see it — that is
exactly what gating exists to catch.

Source: `run_with_dataset()` + `plot_dataset_results()` in
`ch4_rf_point_positioning/example_comparison.py` (`--data ch4_rf_2d_nlos`).

In [ ]:
from ch4_rf_point_positioning.example_comparison import run_with_dataset, plot_dataset_results

DATASET = "ch4_rf_2d_nlos"  # deck: beacons 1 & 2 carry +0.8 m range / +18 deg bearing bias

results = run_with_dataset(f"data/sim/{DATASET}", verbose=False)
fig = plot_dataset_results(results)  # output_file=None: builds and returns, writes nothing
plt.show()


## p32 — Build the map once, query it forever

**Deck claim (p32):** fingerprinting has two stages that never repeat work — OFFLINE, walk
every grid point once and log the RSS from every AP into a database; ONLINE, query that
database against a live reading, as many times as you like, without walking the room again.

Source: `load_fingerprint_database()` + `nearest_neighbor_localize()` /
`k_nearest_neighbor_localize()` in `core/fingerprinting/`, the same primitives
`ch5_fingerprinting/example_deterministic.py` and `example_walk_posterior.py` both build
on.

In [ ]:
from core.fingerprinting import (
    load_fingerprint_database,
    nearest_neighbor_localize,
    k_nearest_neighbor_localize,
)

DATA_DIR = "data/sim/ch5_wifi_fingerprint_grid"
FLOOR_ID = 0
K = 3

db = load_fingerprint_database(DATA_DIR)  # the offline survey, built once
print(db)

# ONLINE: query the map above -- as many times as you like, no new walking required.
# db.features is (M, N) for a single-sample survey, (M, S, N) for a repeat one.
feats = db.features if db.features.ndim == 2 else db.features[:, 0, :]
query, true_loc = feats[0], db.locations[0]

x_nn = nearest_neighbor_localize(query, db, metric="euclidean", floor_id=FLOOR_ID)
x_knn = k_nearest_neighbor_localize(query, db, k=K, metric="euclidean", floor_id=FLOOR_ID)

print(f"\none query against the map above:")
print(f"  true location    {true_loc}")
print(f"  NN estimate      {x_nn}")
print(f"  {K}-NN estimate     {x_knn}")


## p33 — Six matchers, one database — the matcher is a knob

**Deck claim (p33):** k-NN(k=3) leads at every noise level — 3.24 m at baseline, 8.94 m
under 5x the noise; linear regression is worst throughout — 6.13 m at baseline, 11.58 m at
high noise. Every method degrades together as noise rises: the matcher is a knob, not a
fix.

Source: `ch5_fingerprinting/example_comparison.py`'s `main()` — six matchers (NN, k-NN,
MAP, Posterior Mean, Posterior Mean top-k, Linear Regression) against the SAME database
and the SAME three noise scenarios (baseline `noise_std=1.0`, moderate `2.0`, high `5.0`
dBm, each its own seed). Called directly (not re-implemented): `main()` takes no
notebook-unfriendly state, so this cell patches `sys.argv` to the bare program name and
calls it, which is what running `python -m ch5_fingerprinting.example_comparison` with no
flags does.

In [ ]:
import sys

from ch5_fingerprinting import example_comparison as ch5_comparison

_argv = sys.argv
sys.argv = ["example_comparison.py"]  # so argparse sees no flags -> deck-exact defaults
try:
    ch5_comparison.main()
finally:
    sys.argv = _argv


## p37 — DR is always required

**Deck claim (p37):** a synthetic 30x20 m walk (seed 42, 120 s, 100 m of total ground
track), simulated with a consumer-grade MEMS IMU. Raw integration alone drifts badly;
ZUPT, wheel odometry and magnetometer-aided PDR each bound it in a different way.

Source: `ch6_dead_reckoning/example_comparison.py` — `generate_mixed_trajectory` +
`add_sensor_noise` build the scenario, `run_imu_only` / `run_imu_zupt` / `run_wheel_odom`
/ `run_pdr` are the four methods, `plot_comparison` builds the figure and the RMSE table.
`plot_comparison` closes its own figures after saving them (a script-friendly habit), so
this cell reads the saved PNGs back for inline display rather than re-opening them by
hand.

In [ ]:
from IPython.display import Image, display

from core.sensors import FrameConvention, IMUNoiseParams, NavStateQPVP
from ch6_dead_reckoning.example_comparison import (
    DEFAULT_SEED,
    LEVER_ARM_A,
    generate_mixed_trajectory,
    add_sensor_noise,
    run_imu_only,
    run_imu_zupt,
    run_wheel_odom,
    run_pdr,
    plot_comparison,
)

DURATION_S = 120.0  # deck: "120 s walk"
DT_S = 0.01         # 100 Hz
HEIGHT_M = 1.75
SEED = DEFAULT_SEED  # 42, deck-exact

frame = FrameConvention.create_enu()
imu_params = IMUNoiseParams.consumer_grade()

(t, pos_true, vel_true, accel_body, gyro_body, heading_true, mag_body,
 stance, wheel_true) = generate_mixed_trajectory(DURATION_S, DT_S, frame, lever_arm_a=LEVER_ARM_A)
accel_meas, gyro_meas, mag_meas, wheel_meas = add_sensor_noise(
    accel_body, gyro_body, mag_body, wheel_true, DT_S, imu_params, seed=SEED
)

initial = NavStateQPVP(q=np.array([1.0, 0.0, 0.0, 0.0]), v=vel_true[0], p=pos_true[0])
methods = {
    "IMU Only": run_imu_only(t, accel_meas, gyro_meas, initial, frame),
    "IMU + ZUPT": run_imu_zupt(t, accel_meas, gyro_meas, initial, frame, imu_params)[0],
    "Wheel Odom": run_wheel_odom(t, wheel_meas, gyro_meas, initial, LEVER_ARM_A),
    "PDR (Mag)": run_pdr(t, accel_meas, mag_meas, HEIGHT_M)[0],
}

figs_dir = Path(os.environ["IPIN_FIGS_DIR"]) / "ch6_p38"
metrics = plot_comparison(t, pos_true, methods, figs_dir)
for name, m in metrics.items():
    print(f"{name:<14} RMSE {m['rmse']:>6.2f} m   final {m['final']:>6.2f} m   path {m['path']:>7.1f} m")

for stem in ("comparison_trajectories", "comparison_error_time", "comparison_error_cdf"):
    display(Image(filename=str(figs_dir / f"{stem}.png")))


## p39 — Demo from IPIN book's example — same walk, two headings

**Deck claim (p39):** on a pre-generated 124 m corridor walk (170 true steps, seed 42),
gyro-integrated heading drifts to 4.8 m RMSE; magnetometer-aided heading (level assumption)
stays bounded at 1.2 m. 169 of 170 steps are detected.

Source: `run_with_dataset()` in `ch6_dead_reckoning/example_pdr.py`
(`--data ch6_pdr_corridor_walk`), over the repository's own shipped
`data/sim/ch6_pdr_corridor_walk` dataset.

In [ ]:
from core.utils import resolve_data_path
from ch6_dead_reckoning.example_pdr import run_with_dataset

DATASET = "ch6_pdr_corridor_walk"  # deck: "169 of 170 steps detected"
HEIGHT_M = 1.75
LATITUDE_DEG = 45.0
STEP_MODEL = "book"  # Eq. (6.49)

data_path = resolve_data_path(Path("data/sim") / DATASET)
run_with_dataset(str(data_path), height=HEIGHT_M, lat_deg=LATITUDE_DEG, step_model=STEP_MODEL)


## p57 — In Chapter 7 — scan matching finds the motion

**Deck claim (p57):** the true motion between two scans is `dx=0.60 m, dy=-0.40 m,
dyaw=8.0 deg`, unknown to the solver — ICP recovers it from point correspondences alone
(Eqs. 7.10-7.11).

Source: `plot_icp_correspondences()` in
`ch7_slam/example_scan_matching_visualization.py`. **Runtime note:** this file's own
`main()` builds four figures (ICP, NDT voxels, an NDT score surface and a convergence
basin) end to end in ~131 s; this cell calls only the one this slide needs.

In [ ]:
from ch7_slam.example_scan_matching_visualization import (
    TRUE_MOTION,
    plot_icp_correspondences,
)

print(
    f"true motion between scans: dx={TRUE_MOTION[0]:.2f} m, dy={TRUE_MOTION[1]:.2f} m, "
    f"dyaw={np.degrees(TRUE_MOTION[2]):.1f} deg"
)

fig = plot_icp_correspondences(max_pairs=45)
plt.show()


## p58 — Robotics uses LiDAR and Camera

**Deck claim (p58):** a 10-pose straight-line trajectory — raw odometry drifts to 0.137 m;
the scan-matching front end cuts that to an RMSE of 0.0154 m (a 90.2% reduction).

Source: `run_frontend_demo()` + `build_figure()` in `ch7_slam/example_slam_frontend.py`.

In [ ]:
from ch7_slam.example_slam_frontend import run_frontend_demo, build_figure

N_POSES = 10  # deck: "10 poses on a straight line"
SEED = 42

demo = run_frontend_demo(n_poses=N_POSES, seed=SEED)
fig = build_figure(demo)
plt.show()


## p61 — Ch. 7 — LiDAR odometry: match, then re-solve

**Deck claim (p61):** odometry alone drifts to 0.85 m RMSE. Scan-to-map matching (the
"frontend") brings that to 0.53 m; pose-graph optimisation over the 147 loop closures it
finds brings the FINAL RMSE to 0.30 m.

*(Note on the deck's own caption: it states "0.85 m odometry alone, 0.53 m" as the
before/after pair. Measured fresh this session, 0.53 m is the frontend/scan-to-map stage,
not the final pose-graph-optimised result — the optimised RMSE is 0.30 m. All three
numbers are printed below so the full chain is visible; this is flagged, not corrected,
in the deck workspace, which is out of scope for this notebook.)*

Source: `run_with_inline_data()` in `ch7_slam/example_pose_graph_slam.py`.
**Runtime note: ~116 s** — measured, the slowest cell in this notebook (145 poses, 147
loop closures verified by observation).

In [ ]:
from ch7_slam.example_pose_graph_slam import run_with_inline_data

# Deck-exact: the function's OWN default is n_laps=2; the CLI and this repo's
# ch7_slam/QUICK_START.md performance table both use 3 -- pass it explicitly.
run_with_inline_data(trajectory_type="square", n_laps=3)
plt.show()


## p62 — Ch. 7 — bearings only, until bundle adjustment

**Deck claim (p62):** 8 camera poses on a circular trajectory, 6 3D landmarks, 46
bearing-only observations — Eqs. (7.40)-(7.43) (pinhole projection) feed a bundle-adjustment
factor graph, Eqs. (7.68)-(7.70).

Source: `main()` in `ch7_slam/example_bundle_adjustment.py` (a plain function, not an
argparse CLI — `animate` is a real keyword argument here).

In [ ]:
from ch7_slam.example_bundle_adjustment import main as run_bundle_adjustment

run_bundle_adjustment(animate=False)
plt.show()


## Where this notebook stops

This gallery reproduces the deck's own repository-generated figures, in the deck's own
order, through the deck's own citations. Deck pages built from external sources (an
outside paper's figure, a physical-experiment photo, a conceptual diagram with no repo
script behind it) are not here by construction — see this repository's chapter READMEs
and the seven per-chapter notebooks linked from them for the full, guided tour of every
chapter's own examples, including ones this particular talk did not have room for.
